## **Libraries**

In [ ]:
import os
import cv2
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from ultralytics import YOLO
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
import plotly.io as pio
pio.renderers.default = "svg"

In [3]:
import warnings
warnings.filterwarnings("ignore")

## **Configurations**

In [ ]:
base_dir = os.path.dirname(os.path.abspath(__file__))
img_folder = os.path.join(base_dir, "data", "train", "images") 
img_paths = random.sample(list(Path(img_folder).iterdir()) , 9)
data = os.path.join(base_dir, "data.yaml")
output = os.path.join(base_dir, "output") 

epochs = 100
img_size = 640
batch = 16
lr = 1e-2
device = 0
workers = 4
patience = 10
optimizer = "auto"
seed = 42

## **Visualizing Random Sample**

In [ ]:
fig = make_subplots(rows=3, 
                    cols=3,
                    horizontal_spacing=0.08,
                    vertical_spacing=0.08)

rows, cols = 1, 1
for image in tqdm(img_paths):
    img = np.array(Image.open(image).resize((420,420)))

    fig.add_trace(go.Image(z=img),
                  row=rows, col=cols)

    cols += 1
    if cols > 3:
        rows += 1
        cols = 1

fig.update_layout(title="Random Sample of Images",
                  title_x=0.5,
                  template="plotly_dark",
                  width=1000,
                  height=1000,
                  showlegend=False)    

fig.show()    

## **Training a Baseline Model**

In [ ]:
model = YOLO("yolo11s.pt")

model.train(data= data,
            epochs= epochs,
            imgsz= img_size,
            batch= batch,
            lr0 = lr,
            project= str(output),
            name= "Baseline",
            exist_ok= True,
            seed= seed,
            patience= patience,
            device= device,
            workers= workers,
            pretrained= True,
            save= True,
            plots= True)

## **Validation**

In [ ]:
trained_model = Path.cwd() / "Baseline" / "weights" / "best.pt"
baseline_model = YOLO(trained_model)
metrics = baseline_model.val(data= data,
                        imgsz= img_size,
                        device= device,
                        verbose= False)

In [ ]:
res = {
    "Name" : "Baseline Model",
    "mAP50" : round(metrics.box.map50, 4),
    "mAP50-95" : round(metrics.box.map, 4),
    "Precision" : round(metrics.box.mp, 4),
    "Recall" : round(metrics.box.mr, 4)
    }

df = pd.DataFrame([res])
df

,Name,mAP50,mAP50-95,Precision,Recall
0,Baseline Model,0.7797,0.4621,0.7808,0.7037


##### **After reviewing the results, the baseline model is sufficient for detecting fire and smoke although it's kinda tricky distinguishing fire from the background but since we're dealing with a persistent frames via a video input and not just a single image, it'll perform well with the right management of the tracking system.**

## **Evaluation On Random Test Images**

In [ ]:
test_imgs_paths = Path.cwd() / "data" / "test" / "images"
test_imgs = random.sample(list(Path(test_imgs_paths).iterdir()), 3)

fig2 = make_subplots(rows=3,
                     cols=2,
                     subplot_titles= [title for i in range(3)
                                      for title in ["Original", "Predictied"]])

for rows, image in enumerate(test_imgs, start= 1):
    img = Image.open(image)
    fig2.add_trace(go.Image(z=img),
                   row=rows,
                   col=1)

    pred = baseline_model.predict(
        str(image),
        conf= 0.25,
        imgsz= 768,
        verbose= False
    )[0]

    pred_result = cv2.cvtColor(
        pred.plot(),
        cv2.COLOR_BGR2RGB
    )

    fig2.add_trace(go.Image(z=pred_result),
                   row=rows,
                   col=2)

fig2.update_xaxes(showticklabels=False, visible=False)
fig2.update_yaxes(showticklabels=False, visible=False)

fig2.update_layout(
    template="plotly_dark",
    height=1200,
    width=1250,
    title="Predicting Random Test Images",
    title_x=0.5,
    margin=dict(t=80)
)

fig2.show()